# Differential KV Cache (DiffKV) A100 GPU Benchmark
This notebook is optimized for running on high-end cloud GPU platforms like **Lightning AI**, **RunPod**, **Vast.ai**, or your own private GPU server equipped with an **NVIDIA A100 GPU (40GB or 80GB VRAM)**.

### Features:
1. **Secure Repository Setup**: Clone your private `Differential-KV` repository without leaving credentials in git logs.
2. **CUDA / Triton Validation**: Run unit tests to verify the Triton combined decode kernel behaves identically to PyTorch reference on the GPU.
3. **High-Performance Benchmark**: Compare DiffKV (compressed) vs Dense PyTorch (exact) across varying context lengths (4K up to 128K) using state-of-the-art models like `Qwen/Qwen2.5-7B-Instruct` or `Qwen/Qwen2.5-14B-Instruct` in native FP16.
4. **Neighborhood Attention Paper Evaluation**: Run custom prompt engineering and answer evaluations with the supplied `nat_paper.txt` context, comparing output quality and efficiency across Low, Mid, and High presets, Early Rank Boost, and Factual Store routing.
5. **128K Context Qwen2.5-14B-Instruct Q4 Evaluation**: Benchmark 4-bit Dense against 4-bit DiffKV on a 128,000-token prompt generated from the Neighborhood Attention paper context to evaluate a large model on a complex prompt at maximum context window.
6. **Interactive Notebook Imports**: Directly import and use `DiffKVHFWrapper` in cells.

## Step 1: Secure Git Clone using GitHub PAT
Enter your GitHub Personal Access Token (PAT) when prompted to clone the repository cleanly without leaving credentials stored in git configs.

In [ ]:
import os
import shutil
import getpass
import urllib.request
import zipfile

# Clean up old directory if it exists
if os.path.exists('Differential-KV'):
    print("Deleting old directory...")
    shutil.rmtree('Differential-KV')

# Ask for PAT securely
token = getpass.getpass("Enter your GitHub Personal Access Token (PAT): ")
url = "https://api.github.com/repos/Omc12/Differential-KV/zipball"

req = urllib.request.Request(url)
req.add_header('Authorization', f'token {token}')
req.add_header('User-Agent', 'python-urllib')

print("Downloading repository zip archive from GitHub...")
try:
    with urllib.request.urlopen(req) as response:
        with open('repo_temp.zip', 'wb') as f:
            f.write(response.read())
            
    print("Extracting...")
    with zipfile.ZipFile('repo_temp.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    
    extracted_dirs = [d for d in os.listdir('.') if os.path.isdir(d) and 'Differential-KV' in d]
    if extracted_dirs:
        os.rename(extracted_dirs[0], 'Differential-KV')
        os.remove('repo_temp.zip')
        print("\nSuccess! Repository extracted into 'Differential-KV'.")
    else:
        print("Could not locate extracted folder.")
except Exception as e:
    print(f"Error: {e}")

Change the current working directory to the cloned repository folder.

In [ ]:
import os
if os.path.exists('Differential-KV'):
    os.chdir('Differential-KV')
print(f"Current working directory: {os.getcwd()}")

## Step 2: Install Dependencies
Install Triton, transformers, accelerate, pytest, and other necessary libraries for CUDA/Triton inference.

In [ ]:
!pip install -q triton psutil tabulate transformers accelerate pytest optimum bitsandbytes

## Step 3: Run Custom Triton Combined Kernel Unit Tests
Verify that the custom Triton sparse attention decode combined kernel matches PyTorch parity tests.

In [ ]:
!python -m pytest ACTIVE_RUNTIME/tests/test_triton_combined.py -v

## Step 4: Run Full Repository Test Suite
Confirm that all other components and integration paths pass successfully.

In [ ]:
!python -m pytest ACTIVE_RUNTIME/tests/ -v

## Step 5: Run Main Comparative Benchmark (DiffKV vs. Dense)
Here we run a comparative benchmark for both `compressed` (DiffKV) and `exact` (Dense PyTorch baseline) modes using `Qwen/Qwen2.5-7B-Instruct` (the default "best model we can" on A100 40GB/80GB).

Context sweep: `4096`, `8192`, `16384`, `32768`, `65536`, `131072` (up to 128K).
Subprocesses are spawned for memory isolation to avoid VRAM accumulation between runs.

In [ ]:
# Sweep context lengths using the measure_active.py script.
# You can change --model to "Qwen/Qwen2.5-14B-Instruct" or "Qwen/Qwen2.5-32B-Instruct" if you have an 80GB VRAM A100.
!python paper/scripts/measure_active.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --ctx 4096 8192 16384 32768 65536 131072 \
    --modes compressed exact \
    --gen 64 \
    --out paper_results.json

## Step 6: Render and Display Results
Load the generated results JSON file and render a beautiful Markdown comparison table.

In [ ]:
import json
from tabulate import tabulate

try:
    with open("paper_results.json") as f:
        data = json.load(f)
    
    results = data.get("results", [])
    
    table_data = []
    for r in results:
        if r.get("status") == "error":
            table_data.append([
                r.get("mode"), r.get("ctx"), "ERROR", "N/A", "N/A", "N/A", "N/A", "N/A"
            ])
            continue
            
        kv = r.get("kv", {})
        store_used = kv.get("store_used_bytes", 0) / 1e9
        dense_full = kv.get("dense_full_bytes", 0) / 1e9
        
        table_data.append([
            r.get("mode"),
            r.get("ctx"),
            f"{r.get('prefill_s', 0):.2f}s",
            f"{r.get('decode_tps', 0):.2f} tokens/s",
            f"{r.get('mx_peak_gb', 0):.2f} GB",
            f"{r.get('mx_decode_peak_gb', 0):.2f} GB",
            f"{store_used:.2f} GB",
            f"{dense_full:.2f} GB"
        ])
        
    headers = [
        "Mode", "Context Length", "Prefill Time", "Decode TPS", 
        "Peak VRAM (Prefill)", "Peak VRAM (Decode)", "KV Used VRAM", "Dense Equiv VRAM"
     ]
    
    print("\n### DiffKV vs Dense PyTorch Benchmark Results ###\n")
    print(tabulate(table_data, headers=headers, tablefmt="github"))
except FileNotFoundError:
    print("Results file 'paper_results.json' not found. Make sure Step 5 completed successfully.")

## Step 7: Run Neighborhood Attention Paper Evaluation Script
Run the specialized `run_nat_eval.py` script. This script loads standard Dense attention and various DiffKV configurations, supplies the custom `nat_paper.txt` context, feeds the two detailed prompts, rates the outputs, and logs metrics like VRAM usage, prefill latency, and TPS.

In [ ]:
!python colab/run_nat_eval.py --model Qwen/Qwen2.5-7B-Instruct

## Step 8: Display the Paper Evaluation Report
Read and display the generated markdown report directly inside the notebook.

In [ ]:
from IPython.display import Markdown, display

report_path = "colab/nat_evaluation_report.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        report_content = f.read()
    display(Markdown(report_content))
else:
    print(f"Report file not found at {report_path}. Ensure Step 7 ran successfully.")

## Step 9: Run Qwen2.5-14B-Instruct (4-bit) 128K Context Evaluation
Run the specialized `run_nat_128k_q4_eval.py` script. This script loads Qwen2.5-14B-Instruct in 4-bit, constructs a 131,072-token prompt using repeated Neighborhood Attention paper text, inserts your custom prompt at the end, and benchmarks standard 4-bit Dense against 4-bit DiffKV.

In [ ]:
!python colab/run_nat_128k_q4_eval.py --model Qwen/Qwen2.5-14B-Instruct

## Step 10: Render and Display 128K Context Evaluation Report
Read and display the generated 128K markdown report directly inside the notebook.

In [ ]:
from IPython.display import Markdown, display

report_path_128k = "colab/nat_128k_q4_report.md"
if os.path.exists(report_path_128k):
    with open(report_path_128k, "r", encoding="utf-8") as f:
        report_content = f.read()
    display(Markdown(report_content))
else:
    print(f"Report file not found at {report_path_128k}. Ensure Step 9 ran successfully.")

## Step 11: Importing and Running DiffKV in Python Cells
You can import DiffKV directly into your notebook cells and run interactive text generation.

In [ ]:
import os
import sys
import torch

# Ensure repository and runtime paths are in sys.path
repo_path = os.getcwd()
active_path = os.path.join(repo_path, "ACTIVE_RUNTIME")
bench_path = os.path.join(repo_path, "benchmarks")

if active_path not in sys.path:
    sys.path.insert(0, active_path)
if bench_path not in sys.path: 
    sys.path.insert(0, bench_path)

diffkv_core_path = os.path.join(active_path, "native_core", "diffkv_core")
if diffkv_core_path not in sys.path:
    sys.path.insert(0, diffkv_core_path)

# Set environment flags
os.environ["DIFFKV_COMPRESSED_DECODE"] = "1"  # Force DiffKV compressed decode path
os.environ["DIFFKV_FACTUAL_STORE"] = "0"      # Disable factual store for standard prose

# Import wrapper
from serving.hf_diffkv_wrapper import DiffKVHFWrapper

# Load model
model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {model_id} on A100 GPU...")

config = {
    "mode": "fp16",
    "block_size": 256,
    "rank": 16,
    "micro_block_size": 256,
    "preset": "mid",
    "serving_mode": "balanced"
}

wrapper = DiffKVHFWrapper(
    model_id=model_id,
    config=config,
    torch_dtype=torch.float16,
    device="cuda:0"
)
wrapper.ensure_loaded()

# Run generation
prompt = "Write a short summary on why Differential KV caches are essential for scaling LLM context lengths."
print(f"\nPrompt: {prompt}\n")
print("Generating...")

output = wrapper.generate(
    prompt=prompt,
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.9
)

print("\n--- Response ---")
print(output)

# Cleanup VRAM
wrapper.stop()
del wrapper
torch.cuda.empty_cache()